# Named Entity Recognition with BiLSTM + Linear-Chain CRF

This cleaned notebook delegates reusable logic to `src/` and avoids the architecture/documentation mismatch found in the original notebook.

## Important audit result

The supplied `.h5` model is a BiLSTM with token-wise softmax and categorical cross-entropy. It is preserved as a transparent baseline. The cells below train the repository's actual CRF implementation.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

In [ ]:
from src.config import MODEL_DIR, OUTPUT_DIR, TrainingConfig
from src.data_preprocessing import load_huggingface_conll2003, dataset_statistics
from src.model_training import train_model
from src.visualization import plot_training_history

## Load the same CoNLL-2003 source used by the original notebook

In [ ]:
train_sentences, label_names = load_huggingface_conll2003('train')
validation_sentences, _ = load_huggingface_conll2003('validation')
test_sentences, _ = load_huggingface_conll2003('test')
label_names, dataset_statistics(train_sentences)

## Train the true BiLSTM-CRF

CRF training can take time on CPU. Adjust epochs or units for experimentation.

In [ ]:
config = TrainingConfig(max_sequence_length=124, embedding_dim=100, lstm_units=128, dense_units=64, epochs=15, batch_size=32)
model, history, word_to_index, tag_to_index = train_model(train_sentences, validation_sentences, MODEL_DIR, OUTPUT_DIR, config, tag_order=label_names)

In [ ]:
plot_training_history(history, OUTPUT_DIR / 'training_curve.png')

## Evaluate

Use `python scripts/evaluate_model.py` to generate seqeval entity precision, recall, F1, token metrics, confusion matrix, and error analysis.

## Run the demo

```bash
streamlit run app/streamlit_app.py
```